# Impact of Fitness Goals on Health Outcomes
### Exploratory Data Analysis
**Study questions:**
1. Does setting fitness goals in Google Fit positively impact health outcomes?
2. How do user behaviours compare to their pre-goal baseline?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Libraries loaded')

In [ ]:
# Load datasets  (run from project root)
import os
# Works whether you run from project root or Scripts/
base = os.path.dirname(os.path.abspath('')) if os.path.basename(os.getcwd()) == 'Scripts' else os.getcwd()

participants = pd.read_csv(os.path.join(base, 'Raw_Data', 'participants.csv'))
daily        = pd.read_csv(os.path.join(base, 'Raw_Data', 'daily_metrics.csv'), parse_dates=['date'])
weekly       = pd.read_csv(os.path.join(base, 'Raw_Data', 'weekly_summary.csv'))

print(f'Participants : {len(participants)} rows')
print(f'Daily metrics: {len(daily)} rows')
print(f'Weekly summary: {len(weekly)} rows')

## 1. Dataset Overview

In [ ]:
participants.head()

In [ ]:
daily.head()

In [ ]:
# Study group sizes
goal_counts = participants['goal_set'].value_counts()
print('Goal-setters :', goal_counts[True])
print('Control group:', goal_counts[False])
print()
print('Goal types breakdown:')
print(participants[participants['goal_set']]['goal_type'].value_counts())
print()
print('Age range:', participants['age'].min(), '–', participants['age'].max())
print('Gender split:')
print(participants['gender'].value_counts())

## 2. Pre vs Post Goal — Key Metrics

In [ ]:
# Merge goal_set into daily
daily = daily.merge(participants[['participant_id','goal_set','fitness_level','age','gender']], on='participant_id')

metrics = ['steps','active_minutes','calories_burned','resting_heart_rate','sleep_hours','weight_kg']

summary = daily.groupby(['goal_set','phase'])[metrics].mean().round(2)
summary

In [ ]:
# % change from pre to post for goal-setters vs control
pre_gs   = summary.loc[(True,  'pre_goal')]
post_gs  = summary.loc[(True,  'post_goal')]
pre_ctrl = summary.loc[(False, 'pre_goal')]
post_ctrl= summary.loc[(False, 'post_goal')]

change = pd.DataFrame({
    'Goal-setters % change' : ((post_gs  - pre_gs)  / pre_gs  * 100).round(1),
    'Control % change'      : ((post_ctrl- pre_ctrl) / pre_ctrl * 100).round(1),
})
print('Pre → Post percentage change by group:')
change

## 3. Weekly Step Trend — Goal-Setters vs Control

In [ ]:
trend = weekly.groupby(['study_week','goal_set'])['steps'].mean().unstack()
trend.columns = ['Control', 'Goal-setters']

fig, ax = plt.subplots()
trend['Goal-setters'].plot(ax=ax, color='steelblue', label='Goal-setters', linewidth=2)
trend['Control'].plot(ax=ax, color='salmon', label='Control', linewidth=2)
ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Goal set (day 0)')
ax.set_xlabel('Study week')
ax.set_ylabel('Avg daily steps')
ax.set_title('Weekly Average Steps: Goal-Setters vs Control')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 4. Resting Heart Rate Over Time

In [ ]:
hr_trend = weekly.groupby(['study_week','goal_set'])['resting_heart_rate'].mean().unstack()
hr_trend.columns = ['Control', 'Goal-setters']

fig, ax = plt.subplots()
hr_trend['Goal-setters'].plot(ax=ax, color='steelblue', label='Goal-setters', linewidth=2)
hr_trend['Control'].plot(ax=ax, color='salmon', label='Control', linewidth=2)
ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Goal set (day 0)')
ax.set_xlabel('Study week')
ax.set_ylabel('Avg resting heart rate (bpm)')
ax.set_title('Resting Heart Rate: Goal-Setters vs Control')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Weight Change Over Time

In [ ]:
wt_trend = weekly.groupby(['study_week','goal_set'])['weight_kg'].mean().unstack()
wt_trend.columns = ['Control', 'Goal-setters']

fig, ax = plt.subplots()
wt_trend['Goal-setters'].plot(ax=ax, color='steelblue', label='Goal-setters', linewidth=2)
wt_trend['Control'].plot(ax=ax, color='salmon', label='Control', linewidth=2)
ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Goal set (day 0)')
ax.set_xlabel('Study week')
ax.set_ylabel('Avg weight (kg)')
ax.set_title('Average Weight Over Time')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Goal Achievement Rate (Goal-Setters Only)

In [ ]:
gs_post = daily[(daily['goal_set'] == True) & (daily['phase'] == 'post_goal')]

achievement = gs_post.groupby('goal_type')['goal_achieved'].mean().round(3) * 100
achievement = achievement.reset_index()
achievement.columns = ['Goal Type', 'Achievement Rate (%)']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(achievement['Goal Type'], achievement['Achievement Rate (%)'], color='steelblue')
ax.set_xlabel('Achievement Rate (%)')
ax.set_title('Daily Goal Achievement Rate by Goal Type')
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 7. Pre vs Post Comparison by Fitness Level

In [ ]:
fl_summary = daily[daily['goal_set'] == True].groupby(['fitness_level','phase'])['steps'].mean().unstack()
fl_summary.columns = ['Post-goal', 'Pre-goal']
fl_summary = fl_summary.reindex(['Low','Moderate','High'])

fl_summary.plot(kind='bar', color=['steelblue','lightgrey'], edgecolor='white')
plt.title('Avg Steps Pre vs Post Goal — by Fitness Level (Goal-Setters)')
plt.ylabel('Avg daily steps')
plt.xlabel('Fitness Level')
plt.xticks(rotation=0)
plt.legend(title='')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()